# SVG Patch Lab — Full Evaluation (Colab Pro / A100)

Runs **both** experiments in sequence:
1. `skeleton_patch_vision` — core thesis (LLM + vision context)
2. `gnn_patch` — experimental (GNN + vision context)

Expected time on A100: ~2-3 hours for full 500 cases each.

In [ ]:
# ── STEP 1: Set your GitHub token ────────────────────────────────────────────
GITHUB_TOKEN = "PASTE_YOUR_TOKEN_HERE"   # <-- fill this in
GITHUB_USER  = "smerarawal"
REPO_NAME    = "EditSVG-patch-lab"

assert GITHUB_TOKEN != "PASTE_YOUR_TOKEN_HERE", "Set GITHUB_TOKEN above!"
print(f"✓ Credentials set for {GITHUB_USER}/{REPO_NAME}")

In [ ]:
# ── STEP 2: Clone repo and install dependencies ───────────────────────────────
import subprocess, sys, os
from pathlib import Path

os.chdir("/content")

if not Path(REPO_NAME).exists():
    clone_url = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_USER}/{REPO_NAME}.git"
    subprocess.run(["git", "clone", "--recurse-submodules", clone_url], check=True)
else:
    os.chdir(REPO_NAME)
    subprocess.run(["git", "pull"], check=True)
    os.chdir("..")

os.chdir(REPO_NAME)
REPO_ROOT = Path.cwd()
print("Repo root:", REPO_ROOT)

# Install dependencies
subprocess.run([
    sys.executable, "-m", "pip", "install",
    "vllm", "cairosvg>=2.7", "Pillow>=9",
    "torch-geometric", "sentence-transformers",
    "pydantic", "openai", "--quiet"
], check=True)

# Install local svgpatchlab package
subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".", "--quiet"], check=True)

print("✓ All dependencies installed!")

In [ ]:
# ── STEP 3: Boot vLLM server ─────────────────────────────────────────────────
import time, urllib.request, urllib.error

os.chdir(REPO_ROOT)

print("Starting Qwen3.5-4B vLLM server...")
vllm_cmd = [
    sys.executable, "-m", "vllm.entrypoints.openai.api_server",
    "--model", "Qwen/Qwen3.5-4B",
    "--port", "8000",
    "--max-model-len", "32768",
    "--gpu-memory-utilization", "0.95",
    "--tensor-parallel-size", "1",
]

import open
vllm_log = open("vllm_server.log", "w")
server = subprocess.Popen(vllm_cmd, stdout=vllm_log, stderr=subprocess.STDOUT)

for tick in range(120):  # wait up to 20 minutes
    if server.poll() is not None:
        raise RuntimeError("vLLM crashed! Check vllm_server.log")
    try:
        with urllib.request.urlopen("http://localhost:8000/v1/models", timeout=2) as r:
            if r.status == 200:
                print(f"✓ vLLM server ready! (took {tick*10}s)")
                break
    except urllib.error.URLError:
        pass
    time.sleep(10)
else:
    raise TimeoutError("vLLM server took too long to start.")

In [ ]:
# ── STEP 4: Run skeleton_patch_vision (CORE THESIS EXPERIMENT) ───────────────
import json
from pathlib import Path

os.chdir(REPO_ROOT)
print("=" * 65)
print(" RUNNING: skeleton_patch_vision (full 500 cases)")
print("=" * 65)

result = subprocess.run([
    sys.executable, "-m", "svgpatchlab.cli", "evaluate",
    "--config", "configs/experiments/skeleton_patch_vision.json",
])

summary_file = REPO_ROOT / "runs/skeleton_patch_vision/summary.json"
if summary_file.exists():
    summary = json.loads(summary_file.read_text())
    overall = summary.get("overall", {})
    print(f"\n{'='*65}")
    print(" SKELETON + VISION RESULTS")
    print(f"{'='*65}")
    print(f"Total Cases       : {overall.get('cases')}")
    print(f"Exact Match Rate  : {overall.get('gold_patch_exact_rate', 0)*100:.2f}%")
    print(f"Valid Output Rate : {overall.get('valid_output_rate', 0)*100:.2f}%")
    by_task = summary.get("by_task", {})
    for task, stats in by_task.items():
        er = stats.get('gold_patch_exact_rate', 0)
        print(f"  {task:20s} exact={er*100:.1f}%  cases={stats.get('cases')}")

In [ ]:
# ── STEP 5: Run gnn_patch (EXPERIMENTAL GNN + VISION) ────────────────────────
os.chdir(REPO_ROOT)
print("=" * 65)
print(" RUNNING: gnn_patch + vision (full 500 cases)")
print("=" * 65)

result = subprocess.run([
    sys.executable, "-m", "svgpatchlab.cli", "evaluate",
    "--config", "configs/experiments/gnn_patch.json",
])

summary_file = REPO_ROOT / "runs/gnn_patch/summary.json"
if summary_file.exists():
    summary = json.loads(summary_file.read_text())
    overall = summary.get("overall", {})
    print(f"\n{'='*65}")
    print(" GNN + VISION RESULTS")
    print(f"{'='*65}")
    print(f"Total Cases       : {overall.get('cases')}")
    print(f"Exact Match Rate  : {overall.get('gold_patch_exact_rate', 0)*100:.2f}%")
    print(f"Valid Output Rate : {overall.get('valid_output_rate', 0)*100:.2f}%")
    by_task = summary.get("by_task", {})
    for task, stats in by_task.items():
        er = stats.get('gold_patch_exact_rate', 0)
        print(f"  {task:20s} exact={er*100:.1f}%  cases={stats.get('cases')}")

In [ ]:
# ── STEP 6: Zip and download results ─────────────────────────────────────────
import zipfile

ZIP_PATH = "/content/final_results.zip"
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    for folder in ["runs/skeleton_patch_vision", "runs/gnn_patch"]:
        p = REPO_ROOT / folder
        if p.exists():
            for f in p.rglob("*"):
                if f.is_file():
                    zf.write(f, f.relative_to(REPO_ROOT))

print(f"✓ Results zipped to {ZIP_PATH}")

# Auto-download in Colab
try:
    from google.colab import files
    files.download(ZIP_PATH)
    print("✓ Download started!")
except ImportError:
    print(f"Not in Colab — find your results at {ZIP_PATH}")

# Shutdown vLLM
server.terminate()
print("✓ Done! vLLM server shut down.")